In [1]:
import pandas as pd
from openpyxl import load_workbook
from openpyxl.styles import PatternFill, Font, Alignment, Border, Side
from openpyxl.utils import get_column_letter

# Load our final RFM data with all segments
df = pd.read_csv(r'C:\Users\Janhavi\ChurnProject\data\processed\telco_rfm.csv')
print("Loaded:", df.shape)


Loaded: (7032, 26)


In [2]:
# Summary 1: By Segment
segment_summary = df.groupby('Segment').agg(
    Total_Customers=('Churn', 'count'),
    Churned=('Churn', 'sum'),
    Avg_Monthly_Charges=('MonthlyCharges', 'mean'),
    Avg_Tenure_Months=('tenure', 'mean'),
    Total_Revenue=('TotalCharges', 'sum')
).round(2).reset_index()
segment_summary['Churn_Rate_%'] = (segment_summary['Churned'] / segment_summary['Total_Customers'] * 100).round(1)

# Summary 2: By Contract Type
contract_summary = df.groupby('Contract').agg(
    Total_Customers=('Churn', 'count'),
    Churned=('Churn', 'sum'),
    Avg_Monthly_Charges=('MonthlyCharges', 'mean'),
    Revenue_Lost=('MonthlyCharges', lambda x: x[df.loc[x.index, 'Churn'] == 1].sum() * 12)
).round(2).reset_index()
contract_summary['Churn_Rate_%'] = (contract_summary['Churned'] / contract_summary['Total_Customers'] * 100).round(1)

# Summary 3: Overall KPIs
total_customers = len(df)
total_churned = df['Churn'].sum()
churn_rate = round(df['Churn'].mean() * 100, 1)
monthly_revenue_lost = round(df[df['Churn'] == 1]['MonthlyCharges'].sum(), 2)
annual_revenue_lost = round(monthly_revenue_lost * 12, 2)

kpi_summary = pd.DataFrame({
    'Metric': ['Total Customers', 'Total Churned', 'Churn Rate %',
               'Monthly Revenue Lost ($)', 'Annual Revenue Lost ($)'],
    'Value': [total_customers, total_churned, f"{churn_rate}%",
              f"${monthly_revenue_lost:,.2f}", f"${annual_revenue_lost:,.2f}"]
})

print("✅ Summaries ready!")
print(kpi_summary)


✅ Summaries ready!
                     Metric          Value
0           Total Customers           7032
1             Total Churned           1869
2              Churn Rate %          26.6%
3  Monthly Revenue Lost ($)    $139,130.85
4   Annual Revenue Lost ($)  $1,669,570.20


In [3]:
output_path = r'C:\Users\Janhavi\ChurnProject\reports\churn_full_report.xlsx'

with pd.ExcelWriter(output_path, engine='openpyxl') as writer:

    # Sheet 1: KPI Overview
    kpi_summary.to_excel(writer, sheet_name='📊 KPI Overview', index=False)

    # Sheet 2: Segment Analysis
    segment_summary.to_excel(writer, sheet_name='👥 By Segment', index=False)

    # Sheet 3: Contract Analysis
    contract_summary.to_excel(writer, sheet_name='📋 By Contract', index=False)

    # Sheet 4: Full raw data
    df.to_excel(writer, sheet_name='📁 Raw Data', index=False)

print("✅ Excel report saved!")
print(f"Location: {output_path}")


✅ Excel report saved!
Location: C:\Users\Janhavi\ChurnProject\reports\churn_full_report.xlsx


In [4]:
import os
size = os.path.getsize(output_path)
print(f"✅ File size: {size:,} bytes")
print("Open it in Excel to see all 4 sheets!")

# Also print what's in the reports folder now
for f in os.listdir(r'C:\Users\Janhavi\ChurnProject\reports'):
    print(f"  📄 {f}")


✅ File size: 670,982 bytes
Open it in Excel to see all 4 sheets!
  📄 churn_full_report.xlsx
  📄 confusion_matrix.png
  📄 feature_importance.png
  📄 model_comparison.png
  📄 sql_analysis.xlsx
